In [2]:
import nba_api
import requests
import pandas as pd
import time as t
from datetime import datetime
import random 
from nba_api.stats.endpoints import boxscoretraditionalv2
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams 

In [2]:
teams.get_teams()

[{'id': 1610612737,
  'full_name': 'Atlanta Hawks',
  'abbreviation': 'ATL',
  'nickname': 'Hawks',
  'city': 'Atlanta',
  'state': 'Georgia',
  'year_founded': 1949},
 {'id': 1610612738,
  'full_name': 'Boston Celtics',
  'abbreviation': 'BOS',
  'nickname': 'Celtics',
  'city': 'Boston',
  'state': 'Massachusetts',
  'year_founded': 1946},
 {'id': 1610612739,
  'full_name': 'Cleveland Cavaliers',
  'abbreviation': 'CLE',
  'nickname': 'Cavaliers',
  'city': 'Cleveland',
  'state': 'Ohio',
  'year_founded': 1970},
 {'id': 1610612740,
  'full_name': 'New Orleans Pelicans',
  'abbreviation': 'NOP',
  'nickname': 'Pelicans',
  'city': 'New Orleans',
  'state': 'Louisiana',
  'year_founded': 2002},
 {'id': 1610612741,
  'full_name': 'Chicago Bulls',
  'abbreviation': 'CHI',
  'nickname': 'Bulls',
  'city': 'Chicago',
  'state': 'Illinois',
  'year_founded': 1966},
 {'id': 1610612742,
  'full_name': 'Dallas Mavericks',
  'abbreviation': 'DAL',
  'nickname': 'Mavericks',
  'city': 'Dallas',

In [1]:
class MysteryBoxscore:
    
    def __init__(self):
        self.teams = teams.get_teams()
        print('MB Instance')

        
    @property
    def get_game_details_ordered(self):
        return self.game_details_ordered
    
    @property
    def get_boxscore(self):
        return self.bxscr
    
    @property
    def get_sample_game_stats(self):
        return self.sample_game
    

    def name_finder(team_id):
        for d in teams:
            if d['id'] == team_id:
                return d[['full_name','team_id']]
        print('No Teams Found')
    
    def nba_dict_to_df(dict_):
        df = pd.DataFrame(dict_['resultSets'][0]['rowSet'])
        df.columns = dict_['resultSets'][0]['headers'] 
        return df

    
    def randomize_game(self,min_season):
        team_id_of_day = teams.get_teams()[random.randint(0,29)]['id'] #Get random team
        games_dict = leaguegamefinder.LeagueGameFinder(team_id_nullable=team_id_of_day).get_dict() #Get Games
        games = MysteryBoxscore.nba_dict_to_df(games_dict)
        sample_game = games[games['GAME_DATE'] >= min_season + '-09-01'].sample()
        self.sample_game = sample_game[['SEASON_ID','GAME_ID','GAME_DATE','MATCHUP','PTS','PLUS_MINUS']].copy()

        return 'Sample Created' #self.sample_game
    
    def pull_boxscore(self):
        self.bxscr = MysteryBoxscore.nba_dict_to_df(boxscoretraditionalv2.BoxScoreTraditionalV2(game_id = self.sample_game['GAME_ID']).get_dict())
        self.bxscr.columns = ['game_id','team_id','team_abbreviation','team_city','player_id','player_name','nickname','start_position',
                              'comment','minutes','fg_made','fg_attempted','fg_percentage','3pt_made','3pt_attempted','3pt_percentage',
                              'ft_made','ft_attempted','ft_percentage','o_reb','d_reb','rebounds','assists','steals','blocks','turnovers','points',
                              'personal_fouls','plus_minus']
        return 'Boxscore Found' #self.bxscr
    
    def get_game_details(self):
        game_details = self.sample_game[['SEASON_ID','GAME_ID','GAME_DATE','MATCHUP','PTS','PLUS_MINUS']].copy()
        game_details['home_team'], game_details['away_team'] = self.bxscr['team_abbreviation'].unique()
        game_details['home_team_id'], game_details['away_team_id'] = self.bxscr['team_id'].unique()
        game_details['away_pts'] = int((game_details['PTS'] - game_details['PLUS_MINUS']).iloc[0])
        game_details.columns =  ['season_id','game_id','game_date','matchup','home_pts','plus_minus','home_team','away_team','home_team_id','away_team_id','away_pts']
        self.game_details_ordered = game_details[['season_id','game_id','game_date','matchup','home_team_id','home_team','home_pts','away_team_id','away_team','away_pts']].copy()
        return 'Game Details summarized'
    
    def get_game(self,min_season):
        self.randomize_game(min_season=min_season)
        self.pull_boxscore()
        self.get_game_details()
        return 'Game successfully pulled'
        
    
    def clean_game_details(self,max_id_int):
        df_gd = self.get_game_details_ordered.copy()
        df_gd['id'] = max_id_int + 1
        df_gd['valid_from'] = datetime.datetime.now()
        df_gd['valid_to'] = None
        self.df_gd_insert = df_gd[['id','valid_from','valid_to','game_id','game_date','home_team',
                                'home_team_id','home_pts','away_team','away_team_id','away_pts']]
        
    def clean_boxscore(self,max_id_int):
        df_bs = self.get_boxscore.copy()
        df_bs['id'] = max_id_int + 1
        df_bs['minutes'] = df_bs['minutes'].str[:2] + df_bs['minutes'].str[-3:]
        df_bs['instance_player_id'] = (df_bs['id'].astype(str) + df_bs['player_id'].astype(str)).astype(int)
        self.df_bs_insert = df_bs[['id','instance_player_id','game_id','team_id','player_id','player_name','start_position',
                                'minutes','fg_made','fg_attempted','fg_percentage','3pt_made','3pt_attempted',
                                '3pt_percentage','ft_made','ft_attempted','ft_percentage','o_reb','d_reb','rebounds','assists',
                                'steals','blocks','turnovers','points','personal_fouls']]
    
    def upload_to_db(self):
        # Fetch variables
        USER = 'postgres.zbxnnskeboyenigadawi'
        PASSWORD = 'HTIIWPDArKFp2fHv'
        HOST = 'aws-0-us-east-2.pooler.supabase.com'
        PORT = 6543
        DBNAME = 'postgres'

        # Construct the SQLAlchemy connection string
        try:
            DATABASE_URL = f"postgresql+psycopg2://{self.USER}:{self.PASSWORD}@{self.HOST}:{self.PORT}/{self.DBNAME}?sslmode=require"
        except:
            print('Connection Failed')  

        # Create the SQLAlchemy engine & session
        engine = create_engine(DATABASE_URL)

        # Get max existent id in game details
        connection = engine.connect()
        max_id = connection.execute(text(f'select max(id) as maxid from public.f2p_nba_game_details'))
        max_id_int = max(x for x in [max_id.fetchone()[0],0] if x is not None)

        # Create dataframe from game details
        self.clean_game_details(max_id_int)
        print('DF Game Details Created succesfully')

        # Upload Data frame Game details
        table_name = 'f2p_nba_game_details'
        try:
            self.df_gd_insert.to_sql(table_name,engine,if_exists = 'append',index=False)
        except Exception as e:
            print(f'Upload Game Details failed: {e}')

        print('DF Game Details successfully uploaded')

        # Update valid to column
        connection.execute(text(f"""
                        UPDATE public.f2p_nba_game_details
                        SET valid_to = (select DISTINCT valid_from from public.f2p_nba_game_details where id = {max_id_int})
                        WHERE valid_to is NULL and id != {max_id_int};
                        """))
        connection.commit()
        connection.close()
        # Create dataframe from box score
        self.clean_boxscore(max_id_int)
        print('DF Boxscore Created succesfully')
        
        

        # Upload Data frame Box score
        table_name = 'f2p_nba_boxscores'
        try:
            self.df_bs_insert.to_sql(table_name,engine,if_exists = 'append',index=False)
        except Exception as e:
            print(f'Upload Boxscore failed: {e}')

        print('DF Boxscore successfully uploaded')



In [142]:
MB = MysteryBoxscore()
MB.get_game('2015')
MB.upload_to_db()

'Game successfully pulled'

In [144]:
# Connect to the database
try:
    connection = psycopg2.connect(
        user=USER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME
    )
    print("Connection successful!")
    
    # Create a cursor to execute SQL queries
    cursor = connection.cursor()
    
    # Example query
    cursor.execute(f"""
                   UPDATE public.f2p_nba_game_details
                   SET valid_to = (select DISTINCT valid_from from public.f2p_nba_game_details where id = 2)
                   WHERE id = 1;
                   """)
    #print("Current Time:", result)

    # Close the cursor and connection
    cursor.close()
    connection.commit()
    connection.close()
    print("Connection closed.")

except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!
Connection closed.


## Sofascore Swiss Super League Scraping

In [2]:
import requests

In [3]:
import datetime
import json
from bs4 import BeautifulSoup 
import unicodedata
import uuid
from pandasql import sqldf
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
class SofaScore_Stats:

    def __init__(self):
        print('Initialize SofaScore Stats Instance')

    def get_creds(self):
        self.USER = 'postgres.zbxnnskeboyenigadawi'
        self.PASSWORD = 'facba1-wudzEb-fujbux'
        self.HOST = 'aws-0-us-east-2.pooler.supabase.com'
        self.PORT = 6543
        self.DBNAME = 'postgres'

    def query_season_id(self,season):
        #Get creds
        self.get_creds()

        # Construct the SQLAlchemy connection string
        try:
            DATABASE_URL = f"postgresql+psycopg2://{self.USER}:{self.PASSWORD}@{self.HOST}:{self.PORT}/{self.DBNAME}?sslmode=require"
        except:
            print('Connection Failed')  

        # Create the SQLAlchemy engine & session
        engine = create_engine(DATABASE_URL)

        # Get max existent id in game details
        connection = engine.connect()
        self.season_id = connection.execute(
            text(f"""select DISTINCT id from stats_hub.tournaments
                    where name = '{season}' """)
            ).fetchone()[0]
        connection.close()
        

    def get_games_in_round(self,season,rounds:list):
        self.query_season_id(season)
        round_info = []
        now = datetime.datetime.now()
        for round in rounds:
            print(round)
            response = requests.get(f'https://www.sofascore.com/api/v1/unique-tournament/215/season/{self.season_id}/events/round/{round}')
            for game in response.json()['events']:
                game_id = game['id']
                round_info.append([game['id'],
                                game['season']['id'],
                                game['roundInfo']['round'],
                                game['homeTeam']['name'],
                                game['homeTeam']['id'],
                                game['awayTeam']['name'],
                                game['awayTeam']['id'],
                                datetime.datetime.fromtimestamp(game['startTimestamp']),
                                json.dumps(game['homeScore']),
                                json.dumps(game['awayScore']),
                                now
                                ]
                                )
                
            print(f'''Round {round} obtained''')

        self.round_df = pd.DataFrame(round_info)
        self.round_df.columns = ['game_id','tournament_id','round','home_team','home_team_id','away_team',
                                 'away_team_id','start_timestamp_et','home_team_statistics','away_team_statistics','updated_at']

    def upload_games(self, rounds: tuple,check_replace = True):
        self.get_creds()
        # Construct the SQLAlchemy connection string
        try:
            DATABASE_URL = f"postgresql+psycopg2://{self.USER}:{self.PASSWORD}@{self.HOST}:{self.PORT}/{self.DBNAME}?sslmode=require"
        except:
            print('Connection Failed')  
        engine = create_engine(DATABASE_URL)

        if check_replace == True:
            connection = engine.connect()
            connection.execute(text(f"""
                        DELETE from stats_hub.game_instances
                        WHERE tournament_id = {self.season_id} and round in {rounds}
                        """))
            connection.commit()
            connection.close()
        
        table_name = 'game_instances'
        try:
            self.round_df.to_sql(table_name,engine,if_exists = 'append',index=False, schema = 'stats_hub')
        except Exception as e:
            print(f'Upload Game Details failed: {e}')
        print(f'Games successfully uploaded to {table_name}')
    
    def get_stats(self,game_id:list):
        now = datetime.datetime.now()
        player_stats = []
        for game in game_id:
            response_game = requests.get(f'https://www.sofascore.com/api/v1/event/{game}/lineups')
            for side in ['home','away']:
                try:
                    for player in response_game.json()[side]['players']:
                        player_stats.append(
                            [
                                int(str(game) + str(player['player']['id'])),
                                game,
                                player['teamId'],
                                side,
                                player['player']['name'],
                                player['player']['id'],
                                player['position'],
                                player['substitute'],
                                player['jerseyNumber'],
                                player['player']['country']['name'],
                                json.dumps(player['statistics']),
                                response_game.json()[side]['formation'],
                                json.dumps(player['player']),
                                now
                            ]
                        )
                except:
                    print('No data available for fixture: continue')
        self.game_df = pd.DataFrame(player_stats)
        self.game_df.columns = ['id','game_id','current_team_id','side','player_name','player_id','position','substitute','player_number',
                                'nationality','game_stats','formation','player_raw_data','updated_at']
        print(f'''Game IDs {game_id} obtained''')

    def upload_stats(self,game_id: tuple, check_replace = True):
        self.get_creds()
        # Construct the SQLAlchemy connection string
        try:
            DATABASE_URL = f"postgresql+psycopg2://{self.USER}:{self.PASSWORD}@{self.HOST}:{self.PORT}/{self.DBNAME}?sslmode=require"
        except:
            print('Connection Failed')  
        engine = create_engine(DATABASE_URL)

        if check_replace == True:
            connection = engine.connect()
            connection.execute(text(f"""
                        DELETE from stats_hub.player_statistics
                        WHERE game_id in {game_id}
                        """))
            connection.commit()
            connection.close()
        
        table_name = 'player_statistics'
        try:
            self.game_df.to_sql(table_name,engine,if_exists = 'append',index=False, schema = 'stats_hub')
        except Exception as e:
            print(f'Upload Game Details failed: {e}')
        
        print(f'Stats successfully uploaded to {table_name}')

    def get_rounds_in_previous(self, days = 7,tournament = 'Super League Switzerland 2024-2025'):
        self.get_creds()
        self.query_season_id(season = tournament)
        try:
            DATABASE_URL = f"postgresql+psycopg2://{self.USER}:{self.PASSWORD}@{self.HOST}:{self.PORT}/{self.DBNAME}?sslmode=require"
        except:
            print('Connection Failed')  
        engine = create_engine(DATABASE_URL)
        connection = engine.connect()
        info = connection.execute(text(f"""
                    SELECT 
                        DISTINCT round, game_id
                    from stats_hub.game_instances
                    WHERE tournament_id = {self.season_id}
                        and start_timestamp_et >= NOW() - INTERVAL '{days} Day' and start_timestamp_et < NOW()
                    """)).fetchall()
        connection.commit()
        connection.close()
        rounds = list(set([a[0] for a in info]))
        game_ids =  list(set([a[1] for a in info]))
        return rounds, game_ids

    def get_sl_teams_fbt(self):
        return {
            'bsc-young-boys':2445,
            'fc-basel':2501,
            'fc-lugano':2443,
            'fc-luzern':2453,
            'servette-fc':2448,
            'fc-lausanne-sport':2446,
            'fc-sankt-gallen-1879':2442,
            'fc-zuerich':2450,
            'fc-sion':2452,
            'yverdon-sport-fc':2460,
            'grasshopper-club-zuerich':2449,
            'fc-winterthur':2458
            }
         
    def clean_name(self,input_str, pattern, no_removal=False):
        if no_removal == False:
            ### only last name
            last_name = input_str[input_str.rfind(pattern) + 1:]
        else:
            last_name = input_str
        ### Remove accents
        last_name = last_name.replace('Đ','Dj') # Special Case

        nfkd_form = unicodedata.normalize('NFKD', last_name)
        name = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
        return name
         
    def get_roster_updates(self,tournament = 'Super League Switzerland 2024-2025'):
        if 'super league' in tournament.lower():
            teams_fbt = self.get_sl_teams_fbt()
        else:
            return ('Invalid Tournament: No Teams list found')
        
        ###### First: Find current roster (to account for transfers and other changes)
        self.rosters_and_injuries = pd.DataFrame()
        for key,val in teams_fbt.items():
            lineups = requests.get(f'''https://www.sofascore.com/api/v1/team/{val}/players''')
            rosters = []
            for player in lineups.json()['players']:
                rosters.append([
                    player['player']['name'],
                    self.clean_name(player['player']['name'],' ',no_removal=True),
                    player['player']['shortName'],
                    self.clean_name(player['player']['shortName'],' '),
                    player['player']['shortName'][0],
                    player['player']['position'],
                    player['player']['id'],
                    val
                ])
            rosters = pd.DataFrame(rosters)
            rosters.columns = ['player_name','player_name_clean','player_shortname','player_shortname_clean','first_initial','position','player_id','team_id']

            ###### Second: Get injury updates from fussballtransfers.com

            injuries = requests.get(f'''https://www.fussballtransfers.com/verein/{key}/kader/''')
            soup = BeautifulSoup(injuries.text,'html.parser')
            players = soup.find_all(attrs={'class': 'personCardCell__infos'})
            roster_injuries = []
            for i in range(0,len(players)):
                roster_injuries.append(
                [self.clean_name(players[i].find(attrs ={'class':'personCardCell__name'}).get_text().strip(),' '),
                players[i].find(attrs ={'class':'personCardCell__name'}).get_text().strip()[0],
                len(players[i].find(attrs ={'class':'personCardCell__description'}).find_all(attrs={'class':'personCardCell__icon personCardCell__icon--redCard'})) > 0,
                len(players[i].find(attrs ={'class':'personCardCell__description'}).find_all(attrs={'class':'personCardCell__icon personCardCell__icon--injury'})) > 0
                ])
            roster_injuries = pd.DataFrame(roster_injuries)
            roster_injuries.columns = ['player_shortname_clean','first_initial','suspension_flag','injury_flag']

            ### Merge everything
            output = sqldf(
                """
                SELECT DISTINCT
                    r.*,
                    coalesce(ri.suspension_flag, ri2.suspension_flag,ri3.suspension_flag) as suspension_flag,
                    coalesce(ri.injury_flag, ri2.injury_flag, ri3.injury_flag) as injury_flag
                FROM rosters r 
                LEFT JOIN roster_injuries ri
                    on (r.player_shortname_clean = ri.player_shortname_clean and r.first_initial = ri.first_initial)
                LEFT JOIN roster_injuries ri2
                    on (lower(r.player_name_clean) like '%' || lower(ri2.player_shortname_clean) || '%') and
                        (lower(r.player_name_clean) like '% ' || ri2.first_initial || '%' or
                            lower(substring(r.player_name_clean,1,1)) = lower(ri2.first_initial))    
                LEFT JOIN roster_injuries ri3 
                    on r.player_shortname_clean = ri3.player_shortname_clean
                """)
            output['suspension_flag'] = output['suspension_flag'].map({0.0:False, 1.0:True, np.nan:np.nan})
            output['injury_flag'] = output['injury_flag'].map({0.0:False, 1.0:True, np.nan:np.nan})
            
            self.rosters_and_injuries = pd.concat([self.rosters_and_injuries,output[['player_name','player_id','team_id','suspension_flag','injury_flag','position']]], ignore_index = True)
        print(f'''Rosters and Injuries succesfully pulled for {tournament}''')

    def upload_rosters_and_injuries(self,tournament = 'Super League Switzerland 2024-2025'):
        self.get_creds()
        now = datetime.datetime.now()
        self.rosters_and_injuries['valid_from'] = now 
        self.rosters_and_injuries['valid_to'] = None
        self.rosters_and_injuries['id'] = self.rosters_and_injuries.apply(lambda x: str(int(now.timestamp())) + str(x['player_id']), axis=1)
        self.rosters_and_injuries = self.rosters_and_injuries[['id','team_id','player_id','player_name',
                                                               'injury_flag','suspension_flag','position','valid_from','valid_to']]

        try:
            DATABASE_URL = f"postgresql+psycopg2://{self.USER}:{self.PASSWORD}@{self.HOST}:{self.PORT}/{self.DBNAME}?sslmode=require"
        except:
            print('Connection Failed')
        engine = create_engine(DATABASE_URL)
        table_name = 'roster_availability'
        try:
            connection = engine.connect()
            connection.execute(text(f"""
                        UPDATE stats_hub.{table_name}
                        SET valid_to = '{now}'
                        WHERE valid_to is NULL;
                        """))
            self.rosters_and_injuries.to_sql(table_name,engine,if_exists = 'append',index=False, schema = 'stats_hub')
            connection.commit()
            connection.close()
            print(f'Games successfully uploaded to {table_name}')
        except Exception as e:
            print(f'Upload Game Details failed: {e}')

    def get_pbp_subinfo(self,incident):
        json_dict = dict()
        if incident['incidentType'] == 'card':
            json_dict.update({
                'player_id' : incident.get('player',{}).get('id'),
                'player_name' : incident.get('player',{}).get('name'),
                'type_card' : incident.get('incidentClass'),
                'reason' : incident.get('reason'),
                'rescinded' : incident.get('rescinded')
            })
        elif incident['incidentType'] == 'substitution':
            json_dict.update({
                'player_id_in' : incident.get('playerIn',{}).get('id'),
                'player_name_in' : incident.get('playerIn',{}).get('name'),
                'player_id_out' : incident.get('playerOut',{}).get('id'),
                'player_name_out' : incident.get('playerOut',{}).get('name'),
                'injury' : incident.get('injury')
            })
        elif incident['incidentType'] == 'injuryTime':
            json_dict.update({
                'addedTime' : incident.get('addedTime')
            })
        elif incident['incidentType'] == 'goal':
            json_dict.update({
                'player_id_goal' : incident.get('player',{}).get('id'),
                'player_name_goal' : incident.get('player',{}).get('name'),
                'player_id_assist' : incident.get('assist1',{}).get('id'),
                'player_name_assist' : incident.get('assist1',{}).get('name'),
                'home_score' : incident.get('homeScore'),
                'away_score' : incident.get('awayScore'),
                'action_breakdown' : incident.get('footballPassingNetworkAction')
            })
        elif incident['incidentType'] == 'varDecision':
            json_dict.update({
                'player_id_check' : incident.get('player',{}).get('id'),
                'player_name_check' : incident.get('player',{}).get('name'),
                'type_check' : incident.get('incidentClass'),
                'confirmed' : incident.get('confirmed')
            })
        return json_dict
    
    def get_event_pbps(self,game_id:list):
        game_output = []
        now = datetime.datetime.now()
        for game in game_id:
            num = 0
            pbp = requests.get(f'https://www.sofascore.com/api/v1/event/{game}/incidents')
            for incident in pbp.json()['incidents']:
                num += 1
                if incident['incidentType'] == 'period':
                    continue
                game_output.append([
                        str(game) + str(num),
                        game,
                        incident.get('incidentType'),
                        incident.get('time'),
                        incident.get('isHome'),
                        json.dumps(self.get_pbp_subinfo(incident)),
                        now])
                
        self.game_pbp = pd.DataFrame(game_output)
        self.game_pbp.columns = ['id','game_id','incident_type','time_mins','isHome','incident_sub_info','updated_at']
        print('Game PBPs successfully pulled')
            
        
    def upload_event_pbps(self,game_id: tuple, check_replace = True):
        self.get_creds()
        # Construct the SQLAlchemy connection string
        try:
            DATABASE_URL = f"postgresql+psycopg2://{self.USER}:{self.PASSWORD}@{self.HOST}:{self.PORT}/{self.DBNAME}?sslmode=require"
        except:
            print('Connection Failed')  
        engine = create_engine(DATABASE_URL)

        if check_replace == True:
            connection = engine.connect()
            connection.execute(text(f"""
                        DELETE from stats_hub.game_events
                        WHERE game_id in {game_id}
                        """))
            connection.commit()
            connection.close()
        
        table_name = 'game_events'
        try:
            self.game_pbp.to_sql(table_name,engine,if_exists = 'append',index=False, schema = 'stats_hub')
        except Exception as e:
            print(f'Upload Game Details failed: {e}')
        
        print(f'PBPs successfully uploaded to {table_name}')

    def refresh_game_and_stats(self,tournament,days = 7):
        #Get rounds and game ids to refresh
        rounds,game_ids = self.get_rounds_in_previous(days = days, tournament = tournament)

        #Refresh rounds
        self.get_games_in_round(season = tournament, rounds = rounds)
        self.upload_games(tuple(rounds + [-1]),check_replace = True)

        #Refresh games
        self.get_stats(game_id = game_ids)
        self.upload_stats(tuple(game_ids+[-1]), check_replace = True)

        #Refresh play by plays
        self.get_event_pbps(game_id = game_ids)
        self.upload_event_pbps(tuple(game_ids+[-1]), check_replace = True)
    
    @property
    def get_season_id(self):
        return self.season_id
    
    @property
    def get_games(self):
        return self.round_df

    @property
    def get_game_stats(self):
        return self.game_df
    
    @property
    def get_rosters_and_injuries(self):
        return self.rosters_and_injuries
    
    @property
    def get_game_pbp(self):
        return self.game_pbp

In [10]:
sf_s = SofaScore_Stats()

Initialize SofaScore Stats Instance


In [11]:
### Weekly Upload (Super League 2024/2025)
sf_s = SofaScore_Stats()
sf_s.refresh_game_and_stats(tournament = 'Super League Switzerland 2024-2025',days = 21)

Initialize SofaScore Stats Instance
32


KeyError: 'events'

In [4]:
response = requests.get(f'https://www.sofascore.com/api/v1/unique-tournament/215/season/8/events/round/32')

In [5]:
response.json()

{'error': {'code': 403, 'reason': 'Forbidden'}}

In [51]:
#Update roster and injury status
import pandas as pd
from pandasql import sqldf
import numpy as np
import sqlalchemy
from sqlalchemy import create_engine
sf_s = SofaScore_Stats()
sf_s.get_roster_updates(tournament = 'Super League Switzerland 2024-2025')
sf_s.upload_rosters_and_injuries(tournament='Super League Switzerland 2024-2025')

Initialize SofaScore Stats Instance
Rosters and Injuries succesfully pulled for Super League Switzerland 2024-2025
Games successfully uploaded to roster_availability


In [ ]:
#Backfill getting games in rounds
rounds = sf_s.get_games_in_round('Super League Switzerland 2024-2025',[x for x in range(1,24)] + [y for y in range(25,34)])
#Uploading games in round
sf_s.upload_games(rounds = (tuple([x for x in range(1,24)]) + tuple([x for x in range(25,34)])), 
                  season = 'Super League Switzerland 2024-2025',
                  check_replace = True)     

In [54]:
### Example of pbp backfill
### Get games and rounds
rounds,game_ids = sf_s.get_rounds_in_previous(days = 7, tournament = 'Super League Switzerland 2024-2025')
### Then get pbps
sf_s.get_event_pbps(game_ids)
### THen upload pbps


Game PBPs successfully pulled


In [55]:
sf_s.upload_event_pbps(tuple(game_ids+[-1]))

PBPs successfully uploaded to game_events


In [32]:
USER = 'postgres.zbxnnskeboyenigadawi'
PASSWORD = 'facba1-wudzEb-fujbux'
HOST = 'aws-0-us-east-2.pooler.supabase.com'
PORT = 6543
DBNAME = 'postgres'

In [286]:
try:
    DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"
except:
    print('Connection Failed')  
    engine = create_engine(DATABASE_URL)

connection = engine.connect()
connection.execute(f"""
                        DELETE from stats_hub.game_instances
                        WHERE tournament_id = 61658 and round in (21,22,34)
                        """)
connection.commit()
connection.close()

,game_id,tournament_id,round,home_team,home_team_id,away_team,away_team_id,start_timestamp,home_team_statistics,away_team_statistics,updated_at
0,13257818,61658,24,FC Lugano,2443,FC Luzern,2453,2025-02-15 12:00:00,"{""current"": 2, ""display"": 2, ""period1"": 0, ""pe...","{""current"": 0, ""display"": 0, ""period1"": 0, ""pe...",2025-02-16 23:24:30.208047
1,13257822,61658,24,FC Sion,2452,FC Zürich,2450,2025-02-15 12:00:00,"{""current"": 2, ""display"": 2, ""period1"": 2, ""pe...","{""current"": 1, ""display"": 1, ""period1"": 1, ""pe...",2025-02-16 23:24:30.208047
2,13257816,61658,24,FC Winterthur,2458,Young Boys,2445,2025-02-15 14:30:00,"{""current"": 1, ""display"": 1, ""period1"": 0, ""pe...","{""current"": 0, ""display"": 0, ""period1"": 0, ""pe...",2025-02-16 23:24:30.208047
3,13257826,61658,24,Yverdon-Sport FC,2460,FC St. Gallen 1879,2442,2025-02-16 08:15:00,"{""current"": 1, ""display"": 1, ""period1"": 0, ""pe...","{""current"": 0, ""display"": 0, ""period1"": 0, ""pe...",2025-02-16 23:24:30.208047
4,13260067,61658,24,Basel,2501,FC Lausanne-Sport,2446,2025-02-16 10:30:00,"{""current"": 1, ""display"": 1, ""period1"": 1, ""pe...","{""current"": 1, ""display"": 1, ""period1"": 0, ""pe...",2025-02-16 23:24:30.208047
5,13260074,61658,24,Grasshopper Club Zürich,2449,Servette FC,2448,2025-02-16 10:30:00,"{""current"": 1, ""display"": 1, ""period1"": 1, ""pe...","{""current"": 2, ""display"": 2, ""period1"": 0, ""pe...",2025-02-16 23:24:30.208047
